In [3]:
import optax
import matplotlib.pyplot as plt
import numpy as np

In [129]:
class Conf:
    max_lrate: float = 1e-2  # absolute maximum learning rate (max in first schedule or in all schedules without decay)
    lrate_span: float = 1e1  # ratio between max and min learning rates in a given schedule
    lrate_decay: float = 0.1  # if < 1, the learning rate decays between schedules
    warmup_fct: float = 0.1  # fraction of schedule epochs to be used for warmup
    opt_steps: int = 1  # Number of steps in the first schedule (they multiply each time in length by opt_mult)
    opt_mult: int = 2  # Multiplier for the number of steps in each schedule (with 2, the number of steps doubles with each new schedule)
    n_epoch: int = 1000

In [130]:
def get_scheduler(conf: Conf) -> optax.Schedule:
    """Get the learning rate scheduler."""
    epochs_per_schedule = np.array([
        conf.opt_steps * (conf.opt_mult**i)
        for i in range(int(conf.n_epoch // conf.opt_steps))
    ])
    n_schedules = (
        np.where(np.cumsum(epochs_per_schedule) > conf.n_epoch)[0][0] + 1
    )
    schedules = [
        {
            'init_value': conf.max_lrate / conf.lrate_span * conf.lrate_decay**i_schedule,
            'peak_value': conf.max_lrate * conf.lrate_decay**i_schedule,
            'warmup_steps': int(
                (conf.opt_steps * (conf.opt_mult**i_schedule))
                * conf.warmup_fct
            ),
            'decay_steps': int(conf.opt_steps * (conf.opt_mult**i_schedule)),
            'end_value': conf.max_lrate
            / conf.lrate_span
            * conf.lrate_decay ** (i_schedule + 1),
        }
        for i_schedule in range(n_schedules)
    ]
    return optax.sgdr_schedule(schedules)

In [131]:
def get_scheduler2(conf: Conf) -> optax.Schedule:
    """Get the learning rate scheduler."""
    epochs_per_schedule = np.array([
        conf.opt_steps * (conf.opt_mult**i)
        for i in range(int(conf.n_epoch // conf.opt_steps))
        if conf.opt_steps * (conf.opt_mult**i) <= conf.n_epoch
    ])
    print(epochs_per_schedule)
    schedules = [
        {
            'init_value': conf.max_lrate / conf.lrate_span * conf.lrate_decay**i_schedule,
            'peak_value': conf.max_lrate * conf.lrate_decay**i_schedule,
            'warmup_steps': int(
                (conf.opt_steps * (conf.opt_mult**i_schedule))
                * conf.warmup_fct
            ),
            'decay_steps': int(conf.opt_steps * (conf.opt_mult**i_schedule)),
            'end_value': conf.max_lrate / conf.lrate_span * conf.lrate_decay**(i_schedule + 1),
        }
        for i_schedule in range(len(epochs_per_schedule))
    ]
    return optax.sgdr_schedule(schedules)

In [132]:
conf = Conf()
scheduler = get_scheduler(conf)
scheduler2 = get_scheduler2(conf)

In [133]:
conf.opt_steps

In [134]:
lrs = [scheduler(i) for i in range(conf.n_epoch)]
lrs2 = [scheduler2(i) for i in range(conf.n_epoch)]

plt.figure(figsize = (10, 5))
plt.scatter(range(conf.n_epoch), lrs)
plt.scatter(range(conf.n_epoch), lrs2)
plt.title("SGD With Warm Restarts Scheduler")
plt.ylabel("Learning Rate")
plt.xlabel("Epochs/Steps")
plt.yscale("log")
plt.xscale("log");